In [ ]:
%%time
from grape import Graph
node_path = "./graphs/rna-kg/nodes.csv"
edge_path = "./graphs/rna-kg/edges.csv"
manually_loaded_RNAKG = Graph.from_csv(
    # Edges related parameters
    ## The path to the edges list csv
    edge_path=edge_path,
    ## Set the comma as the separator between values
    edge_list_separator=",",
    ## The first rows should be used as the columns names
    edge_list_header=True,
    ## The source nodes are in the subject column
    sources_column="subject",
    ## The source nodes are in the object column
    destinations_column="object",
    ## The source nodes are in the subject column
    edge_list_edge_types_column="type",

    # Nodes related parameters
    ## The path to the nodes list csv
    node_path=node_path,
    ## Set the comma as the separator between values
    node_list_separator=",",
    ## The first rows should be used as the columns names
    node_list_header=True,
    ## The column with the node name is the one with name "name".
    nodes_column="name",
    ## The column with the node type is the one with name "type".
    node_list_node_types_column="type",

    # Graph related parameters
    directed=True,
    name="RNA-KG"
)

In [ ]:
%%time
from grape import Graph
node_path = "./graphs/rna-kg/nodes.csv"
edge_path = "./graphs/rna-kg/edges.csv"
manually_loaded_RNAKG_undirected = Graph.from_csv(
    # Edges related parameters
    ## The path to the edges list csv
    edge_path=edge_path,
    ## Set the comma as the separator between values
    edge_list_separator=",",
    ## The first rows should be used as the columns names
    edge_list_header=True,
    ## The source nodes are in the subject column
    sources_column="subject",
    ## The source nodes are in the object column
    destinations_column="object",
    ## The source nodes are in the subject column
    edge_list_edge_types_column="type",

    # Nodes related parameters
    ## The path to the nodes list csv
    node_path=node_path,
    ## Set the comma as the separator between values
    node_list_separator=",",
    ## The first rows should be used as the columns names
    node_list_header=True,
    ## The column with the node name is the one with name "name".
    nodes_column="name",
    ## The column with the node type is the one with name "type".
    node_list_node_types_column="type",

    # Graph related parameters
    directed=False,
    name="RNA-KG-Undirected"
)

In [ ]:
manually_loaded_RNAKG

In [ ]:
manually_loaded_RNAKG.get_maximum_node_degree()

In [ ]:
%%time
train, test = manually_loaded_RNAKG.connected_holdout(train_size=0.7)
train.enable()

In [ ]:
%%time
from grape.embedders import FirstOrderLINEEnsmallen
embedding = FirstOrderLINEEnsmallen().fit_transform(train)

In [ ]:
from grape import GraphVisualizer

vis = GraphVisualizer(
    graph=test,
    support=train,
    n_components=4,
    edge_embedding_methods="Hadamard",
    rotate=True,
    verbose=True,
    # Automatically, since LINE learns a cosine, the visualization tool
    # would dispatch a Cosine-distance based TSNE. This would use the sklearn
    # implementation, which is terribly slow. Therefore, we force it to use the Euclidean distance
    # and therefore the Multicore TSNE implementation (when available).
    decomposition_kwargs=dict(metric="euclidean")
)

In [ ]:
%%time
vis.fit_negative_and_positive_edges(embedding)

In [ ]:
%%time
vis.plot_positive_and_negative_edges()

In [ ]:
# Farlo anche con node2vec skipgram
# provando con diversi valori di p e q (p=q=1, bfs(0.2,5) e dfs like)
# visualizzando con tsne

%%time
from grape.embedders import FirstOrderLINEEnsmallen
embedding = FirstOrderLINEEnsmallen().fit_transform(manually_loaded_RNAKG)

In [ ]:
from grape import GraphVisualizer
visualizer = GraphVisualizer(manually_loaded_RNAKG)

# You can either provide the model name
#visualizer.fit_nodes("DeepWalk SkipGram", library_name="Ensmallen")
# Or provide a precomputed embedding
#
# visualizer.fit_nodes(numpy_array_with_embedding)
# visualizer.fit_nodes(pandas_dataframe_with_embedding)
#
# or alternatively provide the model to be used:
#
# from grape.embedders import DeepWalkSkipGramEnsmallen
# visualizer.fit_nodes(DeepWalkSkipGramEnsmallen())
#
# In this tutorial, we use the embedding we have just computed above:

visualizer.fit_nodes(embedding)

In [ ]:
# And now we can visualize the node types:
visualizer.plot_node_types()

In [ ]:
# too slow, ran for almost an hour without results
from grape.embedders import Node2VecCBOWEnsmallen
embedding_node2vec_cbow = Node2VecCBOWEnsmallen().fit_transform(manually_loaded_RNAKG_undirected)

In [ ]:
from grape import GraphVisualizer
visualizer_node2vec_cbow = GraphVisualizer(manually_loaded_RNAKG_undirected)

visualizer_node2vec_cbow.fit_nodes(embedding_node2vec_cbow)

In [ ]:
# And now we can visualize the node types:
visualizer.plot_node_types()